# GCG Adversarial Suffix Generation — gpt2-xl Surrogate
IIT Bombay EPGD AI/ML Capstone | Red-teaming LLM Trading Agents

Generates 4 GCG adversarial suffixes using gpt2-xl (1.5B) on GPU (or CPU fallback).
Auto-detects GPU capability and installs compatible PyTorch if needed.
Output: `gcg_suffix_cache_gpt2xl.json`

In [ ]:
# Clone repo and set up environment
!git clone -b m/v1 https://github.com/padhnalikhnaseekho/redteam.git
%cd redteam
!pip install -q transformers accelerate

In [ ]:
# Detect GPU capability and choose device
import torch, os

device = 'cpu'
fp16_flag = ''
steps = 100

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram = round(props.total_memory / 1e9, 1)
    print(f'GPU: {props.name}  VRAM: {vram} GB  sm_{props.major}{props.minor}')
    if props.major >= 7:
        # T4 (7.5), A100 (8.0) — supported by PyTorch 2.x
        device = 'cuda'
        fp16_flag = '--fp16'
        steps = 200
        print(f'Using CUDA — {steps} steps with fp16')
    else:
        # P100 (6.0) — PyTorch 2.x dropped sm_60, no Python 3.12 wheel exists for older torch
        print(f'sm_{props.major}{props.minor} incompatible with PyTorch 2.x on Python 3.12 — falling back to CPU')
        print(f'Using CPU — {steps} steps (no fp16)')
else:
    print(f'No GPU — using CPU, {steps} steps')

os.environ['GCG_DEVICE'] = device
os.environ['GCG_FP16'] = fp16_flag
os.environ['GCG_STEPS'] = str(steps)

In [ ]:
import os
device = os.environ.get('GCG_DEVICE', 'cpu')
fp16 = os.environ.get('GCG_FP16', '')
steps = os.environ.get('GCG_STEPS', '100')
cmd = f'python scripts/run_gcg_generate.py --model gpt2-xl {fp16} --device {device} --steps {steps} --out results/gcg_suffix_cache_gpt2xl.json'
print('Running:', cmd)
import subprocess, sys
subprocess.run(cmd.split(), check=True)

In [ ]:
# Optional: also run Llama-2-7b for even better transfer (requires HF token)
# Uncomment and set your token if you want 30-50% transfer rates

# import os
# os.environ['HUGGING_FACE_HUB_TOKEN'] = 'hf_YOUR_TOKEN_HERE'
# !python scripts/run_gcg_generate.py \
#     --model meta-llama/Llama-2-7b-hf \
#     --fp16 \
#     --device cuda \
#     --steps 200 \
#     --out results/gcg_suffix_cache_llama2.json

In [ ]:
# Show results
import json
with open('results/gcg_suffix_cache_gpt2xl.json') as f:
    suffixes = json.load(f)
for key, suffix in suffixes.items():
    print(f'{key}:\n  {suffix}\n')

In [ ]:
# Download the cache file
from IPython.display import FileLink
FileLink('results/gcg_suffix_cache_gpt2xl.json')